# Train genre classifier (collapsed genres)

Pipeline:
1. Clean raw Spotify tracks
2. **Collapse similar genres** into parent labels (pop/rock/electronic/...)
3. Deduplicate tracks (prefer common parents)
4. Hold out **20%** for testing
5. Train and save the model

Prerequisite: `data/raw/dataset.csv`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..").resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

from src.clean import clean_tracks, print_cleaning_report
from src.data_io import AUDIO_FEATURES, TARGET_COLUMN, load_raw_tracks, raw_csv_exists
from src.genre_map import GENRE_COLLAPSE_MAP
from src.train import MODEL_PATH, METRICS_PATH, load_metrics, train_genre_classifier

print("Collapse map size:", len(GENRE_COLLAPSE_MAP))
print("Parent genres:", sorted(set(GENRE_COLLAPSE_MAP.values())))

In [ ]:
assert raw_csv_exists(), "Missing data/raw/dataset.csv — run download/load first."
df_raw = load_raw_tracks()
print("Raw shape:", df_raw.shape)
df_raw[AUDIO_FEATURES + [TARGET_COLUMN]].head()

In [ ]:
df_clean, report = clean_tracks(
    df_raw, min_genre_count=500, collapse_genres=True, save=True
)
print_cleaning_report(report)
df_clean[TARGET_COLUMN].value_counts()

In [ ]:
result = train_genre_classifier(
    df=df_raw, n_estimators=200, min_genre_count=500, collapse_genres=True
)
result["metrics"]["accuracy"], result["metrics"]["n_classes"], result["model_path"]

In [ ]:
metrics = load_metrics()
print("Saved model exists:", MODEL_PATH.exists())
print("Accuracy:", round(metrics["accuracy"], 4))
print("Macro F1:", round(metrics["macro_f1"], 4))
print("Weighted F1:", round(metrics["weighted_f1"], 4))
print("Classes:", metrics["n_classes"])
print("Genres after collapse:", metrics.get("cleaning", {}).get("genres_after_collapse"))
print("Clean genres used:", metrics.get("cleaning", {}).get("clean_genres"))
print("Metrics file:", METRICS_PATH)